## Latent Probing of Specific Positioned Tokens

We are analyzing the embeddings of last input token before model generation and the second last token in all generated tokens. We store the embeddings from all model layers.

The test run was by command `python ../semantic_uncertainty/generate_answers.py --model_name=Mistral-7B-v0.1-4bit --temperature=1 --dataset=svamp --num_samples=200`.

In [30]:
%load_ext autoreload
%autoreload 2

import os
# os.chdir('../slurm/ms23jh/uncertainty/wandb/latest-run/files')
import pickle
import yaml
import json

from matplotlib import pyplot as plt
import pandas as pd
import numpy as np

from copy import deepcopy
import seaborn as sns

import wandb
api = wandb.Api()
api.entity = 'jiatongg'

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [31]:
# Read training generated embeddings
f = open('train_generations.pkl', 'rb')
generations = pickle.load(f)
f.close()

# Read uncertainty measures (p-true, predictive/semantic uncertainties)
g = open('uncertainty_measures.pkl', 'rb')
measures = pickle.load(g)
g.close()

In [50]:
# Basic statistics about generated embeddings
import random
print("Number of generations:", len(generations))
print("Each generation's fields:", list(generations.values())[0].keys())

random_record = list(generations.values())[random.randint(0, len(generations))]
print(random_record['question'])
print(random_record['context'])
print(random_record['most_likely_answer']['emb_last_tok_before_gen'].shape)
# (num_layers+1) * 1 tok * hidden_dim
print(random_record['most_likely_answer']['emb_tok_before_eos'].shape)
print(random_record['reference'])
print(random_record['responses'])

Number of generations: 200
Each generation's fields: dict_keys(['question', 'context', 'most_likely_answer', 'reference', 'responses'])
How many trees does she have left?
Haley grew 9 trees in her backyard. After a typhoon 4 died. Then she grew 5 more trees.
torch.Size([33, 1, 4096])
torch.Size([33, 1, 4096])
{'answers': {'answer_start': [], 'text': ['10.0']}, 'id': 'chal-902'}
[]


In [36]:
# Check out uncertainty measures
measures.keys()

dict_keys(['uncertainty_measures', 'semantic_ids', 'validation_is_false', 'validation_unanswerable', 'alt_validation_accuracies_mean', 'alt_validation_is_false'])

In [57]:
# Check out uncertainty measures
# Use semantic entropy as initial measure 
semantic_entropy = torch.tensor(measures['uncertainty_measures']['semantic_entropy'])
semantic_entropy.shape

torch.Size([200])

In [54]:
# Training a simple MLP as linear probes
import torch
import torch.nn as nn
from torch.nn import Linear, ReLU, Dropout

class MLP(nn.Module):
    def __init__(self, in_dim, hid_dims, out_dim, dropout):
        super(MLP, self).__init__()
        layers = []
        
        # Check if hid_dims is a list or a scalar and construct the layer dimensions list
        if isinstance(hid_dims, int):
            # If it's a scalar, we assume a two-layer network with the same hidden dimension
            layer_dims = [in_dim] + [hid_dims] * 2
        elif isinstance(hid_dims, list):
            # If it's a list, construct layer dimensions starting with in_dim and ending with the last hidden dimension
            layer_dims = [in_dim] + hid_dims
        else:
            raise ValueError("hid_dims must be either an integer or a list of integers")
        
        for i in range(len(layer_dims) - 1):
            layers.extend([
                Linear(layer_dims[i], layer_dims[i+1]),
                ReLU(),
                Dropout(dropout)
            ])
        
        layers.append(Linear(layer_dims[-1], out_dim))        
        self.layers = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.layers(x)

## Explore Last Input Token before Generated Token



In [83]:
tok_bef_gen_dataset = torch.stack([record['most_likely_answer']['emb_last_tok_before_gen'] 
                                   for record in generations.values()]).squeeze(-2).to(torch.float32)
# Originally float64
semantic_entropy = semantic_entropy.to(torch.float32)

print(tok_bef_gen_dataset.shape)

torch.Size([200, 33, 4096])


In [100]:
x = tok_bef_gen_dataset[4]
x.shape, x[2], x[3]

(torch.Size([33, 4096]),
 tensor([ 0.0019, -0.0043, -0.0032,  ...,  0.0003, -0.0019,  0.0030]),
 tensor([-0.0039, -0.0115, -0.0090,  ...,  0.0082, -0.0005,  0.0046]))

### Experment layer-by-layer

In [85]:
layered_datasets = tok_bef_gen_dataset.transpose(0, 1)
print(layered_datasets.shape, layered_datasets.dtype)

torch.Size([33, 200, 4096]) torch.float32


In [86]:
# Make batched inputs and dataloaders (one for each layer)
from torch.utils.data import TensorDataset, DataLoader, random_split

batch_size = 32
train_size = int(200 * 0.8)
test_size = 200 - train_size

train_dataloaders = []
test_dataloaders = []

for i in range(layered_datasets.shape[0]):
    # Extract the i-th dataset from the layered_dataset
    dataset_i = layered_datasets[i]
    
    tensor_dataset_i = TensorDataset(dataset_i, semantic_entropy)
    train_dataset_i, test_dataset_i = random_split(tensor_dataset_i, [train_size, test_size])
    
    train_dataloader_i = DataLoader(train_dataset_i, batch_size=batch_size, shuffle=True)
    test_dataloader_i = DataLoader(test_dataset_i, batch_size=batch_size, shuffle=False)  
    
    train_dataloaders.append(train_dataloader_i)
    test_dataloaders.append(test_dataloader_i)

In [87]:
import torch.optim as optim

criterion = nn.MSELoss()  # Mean Squared Error Loss for entropy regression
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [102]:
# Function to train and evaluate the model (and save the best model for later use)
import copy

def train_and_evaluate(model, train_loader, test_loader, epochs=50):
    criterion = nn.MSELoss() 
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_model_wts = copy.deepcopy(model.state_dict())
    best_loss = float('inf')
    
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        for inputs, scalars in train_loader:  # Replace "_" with your actual labels if available
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, scalars)  # Adjust targets based on your task
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        avg_train_loss = total_train_loss / len(train_loader)
        
        model.eval()
        total_eval_loss = 0
        with torch.no_grad():
            for inputs, scalars in test_loader: 
                outputs = model(inputs)
                loss = criterion(outputs, scalars)  # Adjust targets based on your task
                total_eval_loss += loss.item()
        avg_eval_loss = total_eval_loss / len(test_loader)
        
        # Logging for each epoch
        print(f"Epoch {epoch+1}/{epochs}, Training Loss: {avg_train_loss:.4f}, Evaluation Loss: {avg_eval_loss:.4f}")
        
        # Check if this is the best model so far
        if avg_eval_loss < best_loss:
            best_loss = avg_eval_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            print(f"New best model found at epoch {epoch+1} with Evaluation Loss: {avg_eval_loss:.4f}")
    
    # Load best model weights
    model.load_state_dict(best_model_wts)
    return model, best_loss

In [103]:
# Model definition
in_dim = 4096   # 
hid_dims = [1024, 512, 256]
out_dim = 1    # Output dimension (scalar for semantic entropy)
dropout = 0.5  # Dropout rate

epochs = 50

best_models = []
best_losses = []
for i, (train_loader, test_loader) in enumerate(zip(train_dataloaders, test_dataloaders)):
    model = MLP(in_dim, hid_dims, out_dim, dropout)
    print(f"Training on Dataset {i+1}/{len(train_dataloaders)}")
    best_model, best_loss = train_and_evaluate(model, train_loader, test_loader, epochs)
    best_models.append(best_model)
    best_losses.append(best_loss)

Training on Dataset 1/33
Epoch 1/50, Training Loss: 35.7579, Evaluation Loss: 34.4180
New best model found at epoch 1 with Evaluation Loss: 34.4180
Epoch 2/50, Training Loss: 29.9738, Evaluation Loss: 22.6812
New best model found at epoch 2 with Evaluation Loss: 22.6812
Epoch 3/50, Training Loss: 12.6034, Evaluation Loss: 0.4982
New best model found at epoch 3 with Evaluation Loss: 0.4982
Epoch 4/50, Training Loss: 5.2242, Evaluation Loss: 3.1590
Epoch 5/50, Training Loss: 2.1200, Evaluation Loss: 2.1499
Epoch 6/50, Training Loss: 3.0301, Evaluation Loss: 2.8965
Epoch 7/50, Training Loss: 1.6956, Evaluation Loss: 0.2297
New best model found at epoch 7 with Evaluation Loss: 0.2297
Epoch 8/50, Training Loss: 1.4588, Evaluation Loss: 0.6181
Epoch 9/50, Training Loss: 1.0198, Evaluation Loss: 0.3438
Epoch 10/50, Training Loss: 1.0259, Evaluation Loss: 0.5790
Epoch 11/50, Training Loss: 0.8458, Evaluation Loss: 0.2043
New best model found at epoch 11 with Evaluation Loss: 0.2043
Epoch 12/50

### Test models (on one particular layer) on other layers.

Previously, we only trained and evaluated models within each layer's embedding splits ([200, 4096]). Now we would like to see the linear probe's extrapolation ability to the same token on other layers.

In [104]:
def evaluate_model_on_loader(model, data_loader):
    model.eval()
    criterion = nn.MSELoss()
    total_loss = 0
    with torch.no_grad():
        for inputs, scalars in data_loader: 
            outputs = model(inputs)
            loss = criterion(outputs, scalars)
            total_loss += loss.item()
    avg_loss = total_loss / len(data_loader)
    return avg_loss

# Initialize a matrix to store the cross-dataset losses
cross_dataset_losses = torch.zeros(len(best_models), len(train_dataloaders) + len(test_dataloaders))

for i, model in enumerate(best_models):
    # Evaluate on training datasets
    for j, train_loader in enumerate(train_dataloaders):
        loss = evaluate_model_on_loader(model, train_loader)
        cross_dataset_losses[i, j] = loss
    # Evaluate on test datasets
    for k, test_loader in enumerate(test_dataloaders, start=len(train_dataloaders)):
        loss = evaluate_model_on_loader(model, test_loader)
        cross_dataset_losses[i, k] = loss

# Compare the cross-dataset losses with best_losses for each model
for i, (best_loss, model) in enumerate(zip(best_losses, best_models)):
    print(f"Model {i+1}:")
    print(f"  Best Loss (on its own test set): {best_loss:.4f}")
    for j in range(len(train_dataloaders)):
        print(f"  Loss on Training Dataset {j+1}: {cross_dataset_losses[i, j]:.4f}")
    for k in range(len(test_dataloaders)):
        print(f"  Loss on Test Dataset {k+1}: {cross_dataset_losses[i, len(train_dataloaders) + k]:.4f}")

Model 1:
  Best Loss (on its own test set): 0.1923
  Loss on Training Dataset 1: 0.1264
  Loss on Training Dataset 2: 2.3896
  Loss on Training Dataset 3: 2.4823
  Loss on Training Dataset 4: 0.6567
  Loss on Training Dataset 5: 0.4556
  Loss on Training Dataset 6: 2.4496
  Loss on Training Dataset 7: 6.1823
  Loss on Training Dataset 8: 7.0444
  Loss on Training Dataset 9: 21.7951
  Loss on Training Dataset 10: 46.8496
  Loss on Training Dataset 11: 81.3090
  Loss on Training Dataset 12: 87.3246
  Loss on Training Dataset 13: 88.8054
  Loss on Training Dataset 14: 94.0043
  Loss on Training Dataset 15: 156.4748
  Loss on Training Dataset 16: 221.9111
  Loss on Training Dataset 17: 243.0706
  Loss on Training Dataset 18: 390.9104
  Loss on Training Dataset 19: 318.6948
  Loss on Training Dataset 20: 436.4255
  Loss on Training Dataset 21: 463.4100
  Loss on Training Dataset 22: 375.8350
  Loss on Training Dataset 23: 613.2411
  Loss on Training Dataset 24: 606.8113
  Loss on Training D

#### The further away one layer where the model is trained is from a test layer, the more losses in predicting SE are observed in general. 

#### The last layer is in particular interesting (_because the last layer embeddings were obviously sharpened or scaled up compared to previous ones_; see below); models not trained on it have disproportionately high losses (though it still follows the "distance law").

In [120]:
# Last layer
layered_datasets[-1].mean(), layered_datasets[-1].var()

(tensor(-0.0118), tensor(22.0734))

In [121]:
# Layers except last layer
layered_datasets[:-1].mean(), layered_datasets[:-1].var()

(tensor(-0.0011), tensor(0.0281))

## Explore Second Last Generated Token by Model
